In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
import itertools
from sklearn.cluster import AgglomerativeClustering

from Bio import SeqIO
from Bio import AlignIO
from Bio import Seq
from Bio import SeqRecord
from Bio import GenBank
from Bio.Restriction import *
#from pybedtools import BedTool

from matplotlib import pyplot as plt
from matplotlib import colors
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

import seaborn as sns

import re
import gzip
import allel
import zarr
import colorcet
import progressbar
from collections import Counter

In [ ]:
# Import strain metadata
strains_info = pd.read_csv('strains_info.csv')
strains_info.index = strains_info['strain'].values
base_dir = '/home/mathieu/postdoc_heasley/collaborations/KSO_glabrata/'
fig_path = f'{base_dir}fig/'

In [ ]:
S = ['KSO_138'] + [f'KSO_{i}' for i in range(1, 35) if i not in [5,7]]
strain_alias = dict(zip(S, [int(s[4:]) if s!='KSO_138' else 'CBS138' for s in S]))
strain_color = dict(zip(S, [c+[0.8] for c in colorcet.glasbey_hv[:len(S)]]))
Strain_color = {strain_alias[i]:j for i,j in strain_color.items()}
Strain_order = [strain_alias[s] for s in S]

In [ ]:
qc_summary = pd.read_table('/home/mathieu/data/novogene_feb24/02.Report_X202SC24015069-Z01-F001/src/tables/qc.summary.xls')
qc_summary.index = qc_summary['Sample'].values
qc_summary.drop('Undetermined', inplace=True)
qc_summary['strain_no'] = qc_summary['Sample'].apply(lambda x: int(re.match('(?:KSO_|YJM)(\d+)', x).group(1)))
qc_summary['lab'] = np.where(qc_summary['Sample'].apply(lambda x: x[:4]=='KSO_'), 'KSO', 'LRH')
qc_summary['bases'] = qc_summary['Raw reads'] * 150
qc_summary['approx_cov'] = qc_summary['bases']/12.75e6
# split by lab
qc_summary = qc_summary.loc[qc_summary['lab']=='KSO']

In [ ]:
# Import reference genome
with open(f'{base_dir}data/ref/CBS138.fasta') as fi:
    genome_glabrata = [seq for seq in SeqIO.parse(fi, 'fasta')]

In [ ]:
# Import reference annotations
gff = pd.read_csv(f'{base_dir}data/ref/CBS138.gff', sep='\t', comment='#', header=None)

In [ ]:
# Extract GFF entries corresponding to genes
gff_genes = gff.loc[gff[2]=='gene']
gff_genes = pd.concat([gff_genes.iloc[:, :8], gff_genes[8].apply(lambda x: pd.Series(dict([i.split('=') for i in x.split(';')])))], axis=1)
gff_genes.loc[gff_genes['Alias'].isna(), 'Alias'] = ''
gff_genes.index = gff_genes['ID'].values

In [ ]:
tig_off = pd.Series(dict([(seq.id, len(seq.seq)) for seq in genome_glabrata]), name='len')
tig_off = pd.concat([tig_off, tig_off.cumsum().rename('cumul_end')], axis=1)
tig_off['cumul_start'] = np.concatenate([[0], tig_off['len'].iloc[:-1].cumsum().values])

In [ ]:
centromeres = gff.loc[gff[2]=='centromere'].set_index(0)[[3,4]]
centromeres['loc'] = centromeres.mean(axis=1)
centromeres['Loc'] = centromeres['loc'] + tig_off.loc[centromeres.index, 'cumul_start'].values

# Parse depth data

In [ ]:
DEPTH = []
for s in strains_info.index:
    path = f'../data/depth/{s}.depth_nuc_bin.tab.gz'
    depth = pd.read_csv(path, sep='\t', header=None)
    depth['strain'] = s
    DEPTH.append(depth)
DEPTH = pd.concat(DEPTH).reset_index(drop=True)

depth_median = DEPTH.groupby('strain')[2].median()
DEPTH['depth_rel'] = DEPTH[2]/depth_median.loc[DEPTH['strain']].values

In [ ]:
DEPTH_MT = []
for s in strains_info.index:
    path = f'../data/depth/{s}.depth_mt_bin.tab.gz'
    depth = pd.read_csv(path, sep='\t', header=None)
    depth['strain'] = s
    DEPTH_MT.append(depth)
DEPTH_MT = pd.concat(DEPTH_MT).reset_index(drop=True)

depth_mt_median = DEPTH_MT.groupby('strain')[2].median()
DEPTH_MT['depth_rel'] = DEPTH_MT[2]/depth_median.loc[DEPTH_MT['strain']].values

In [ ]:
# Add coverage data to strains metadata table
strains_info['depth_median'] = depth_median.loc[strains_info.index]
strains_info['depth_mt_median'] = depth_mt_median.loc[strains_info.index]
strains_info['bases'] = qc_summary.loc[strains_info.index, 'bases']

In [ ]:
fig = plt.figure(figsize=[12,10])
gs = plt.GridSpec(ncols=14, nrows=1, width_ratios=tig_off['len'] * np.array([1]*13 + [80]),
                 wspace=0.2, left=0.08, right=0.96, bottom=0.1, top=0.85)
axes = {}

chrom_ax_dict = dict(zip(tig_off.index, range(14)))
S = strains_info.index
wdw_nuc = 5e4
wdw_mt = 200

for chrom, df in DEPTH.loc[DEPTH[0]!='mito_C_glabrata_CBS138'].groupby(0):
    dat = df.pivot_table(index='strain', columns=1, values='depth_rel', aggfunc=lambda x: x)
    dat = dat.loc[S, np.sort(dat.columns)]
    
    ax = fig.add_subplot(gs[chrom_ax_dict[chrom]])

    hm_nuc = ax.imshow(dat, cmap='viridis', vmin=0, vmax=2, aspect='auto', interpolation='None')

    xticks = np.arange(0, dat.shape[1], 5)
    xtick_labels = [f'{x:.0f}' for x in xticks * wdw_nuc * 1e-3]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels, rotation=90, size=8)

    ax.set_title(chr.split('_')[0], size=12)
    
    axes[chr] = ax

chrom = 'mito_C_glabrata_CBS138'
ax = fig.add_subplot(gs[chrom_ax_dict[chrom]])
dat = DEPTH_MT.pivot_table(index='strain', columns=1, values='depth_rel', aggfunc=lambda x: x)
dat = dat.loc[S, np.sort(dat.columns)]

hm_mt = ax.imshow(dat, cmap='magma', vmin=0, vmax=23, aspect='auto', interpolation='None')

xticks = np.arange(0, dat.shape[1], 20)
xtick_labels = [f'{x:.0f}' for x in xticks * wdw_mt * 1e-3]
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels, rotation=90, size=8)

ax.set_title(chrom.split('_')[0], size=12)

axes[chrom] = ax

for chrom, ax in axes.items():
    
    ax.set_yticks(range(len(S)))
    
    if chrom == 'ChrA_C_glabrata_CBS138':    
        ax.set_yticklabels(S)

    else:
        ax.set_yticklabels([])

    for y in np.arange(len(S)-1)+0.5:
        ax.axhline(y, c='w', zorder=2)

    for s in ['top', 'bottom', 'left', 'right']:
        ax.spines[s].set_visible(False)

# Add heatmap colorbar
ax_cbar = fig.add_axes([0.2, 0.93, 0.15, 0.02])
cb = plt.colorbar(hm_nuc, cax=ax_cbar, orientation='horizontal')
cb.outline.set_visible(False)
cb_x_ticks = np.linspace(0, 2, 5)
ax_cbar.text(0.5, 2.5, 'nuc depth of coverage (X)', 
             ha='center', va='top', transform=ax_cbar.transAxes)

ax_cbar = fig.add_axes([0.55, 0.93, 0.15, 0.02])
cb = plt.colorbar(hm_mt, cax=ax_cbar, orientation='horizontal')
cb.outline.set_visible(False)
cb_x_ticks = np.linspace(0, 23, 5)
ax_cbar.text(0.5, 2.5, 'mt depth of coverage (X)', 
             ha='center', va='top', transform=ax_cbar.transAxes)

fig.text(0.5, 0.03, 'Pos (kb)', ha='center', va='bottom')

plt.savefig(f'{fig_path}hm_coverage.png', dpi=300)
#plt.show()
plt.close()

# Population structure

In [ ]:
# from https://alimanfoo.github.io/2015/09/28/fast-pca.html

def plot_ld(gn, title):
    m = allel.rogers_huff_r(gn) ** 2
    ax = allel.plot_pairwise_ld(m)
    ax.set_title(title)

def ld_prune(gn, size, step, threshold=.1, n_iter=1):
    for i in range(n_iter):
        loc_unlinked = allel.locate_unlinked(gn, size=size, step=step, threshold=threshold)
        n = np.count_nonzero(loc_unlinked)
        n_remove = gn.shape[0] - n
        print('iteration', i+1, 'retaining', n, 'removing', n_remove, 'variants')
        gn = gn.compress(loc_unlinked, axis=0)
    return gn

def plot_pca_coords(coords, model, pc1, pc2, ax, sample_population):
    
    global populations
    global pop_mfc
    global pop_mec
    global pop_size
    
    sns.despine(ax=ax, offset=5)
    x = coords[:, pc1]
    y = coords[:, pc2]
    for pop in populations:
        flt = (sample_population == pop)
        
        ax.scatter(x[flt], y[flt], marker='o', label=pop, #linestyle=' ',
                color=pop_mfc[pop], edgecolors=pop_mec[pop], s=pop_size[pop], linewidths=0.5)
    ax.set_xlabel('PC%s (%.1f%%)' % (pc1+1, model.explained_variance_ratio_[pc1]*100))
    ax.set_ylabel('PC%s (%.1f%%)' % (pc2+1, model.explained_variance_ratio_[pc2]*100))
    

def fig_pca(coords, model, title, sample_population=None):
    if sample_population is None:
        sample_population = df_samples.population.values
    # plot coords for PCs 1 vs 2, 3 vs 4
    fig = plt.figure(figsize=(8, 10))
    ax = fig.add_subplot(2, 1, 1)
    plot_pca_coords(coords, model, 0, 1, ax, sample_population)
    ax = fig.add_subplot(2, 1, 2)
    plot_pca_coords(coords, model, 2, 3, ax, sample_population)
    ax.legend(ncols=1, bbox_to_anchor=(1, 0), loc='lower left')
    fig.suptitle(title, y=1.02)
    #fig.tight_layout()

In [ ]:
vcf_path = f'{base_dir}mpileup_all/mpileup_biallelic_snps.vcf.gz'
zarr_path = f'{base_dir}mpileup_all/mpileup_biallelic_snps.zarr'
allel.vcf_to_zarr(vcf_path, zarr_path, fields='*')

## Filter variants

In [ ]:
callset = zarr.open_group(zarr_path, mode='r')
gt = allel.GenotypeChunkedArray(callset['/calldata']['GT'])

In [ ]:
CHROM = callset['/variants']['CHROM'][:]
POS = callset['/variants']['POS'][:]
CHROM_POS = pd.concat([pd.Series(CHROM), pd.Series(POS)], axis=1).apply(lambda x: f'{x[0]}:{x[1]}', axis=1)

In [ ]:
vcf_samples = pd.Series(callset['samples'][:])
vcf_samples_idx = pd.Series(np.arange(vcf_samples.shape[0]), index=vcf_samples)

In [ ]:
f = CHROM == 'ChrD_C_glabrata_CBS138'
dat = pd.DataFrame(gn[f])
dat.index = CHROM_POS[f]
dat.columns = vcf_samples
sns.heatmap(dat)
plt.xticks(np.arange(33)+0.5, vcf_samples)
plt.show()
plt.close()

In [ ]:
# Count missing gt and allele counts
missing = gt.count_missing(axis=1) # 90% of variants with no missing genotype
# convert genotypes no counts of alt observations
gn = gt.to_n_alt()[:]
# extract singletons as variants that have only one alt or ref sample
singletons = ((gn>0).sum(axis=1) == 1) | ((gn>0).sum(axis=1) >= 443) # 15.8% are singletons

In [ ]:
fil = (missing == 0) & (~singletons) # 81% of variants correspond to both criteria

In [ ]:
fil.sum()/fil.shape[0]

In [ ]:
# filter the allele counts matrix with the final filter
gn_fil = gn[fil]

## Prune variants for LD and plot

In [ ]:
#filter_random = np.random.choice(np.arange(gn.shape[0]), 5000, replace=False)
#gnu = gn[filter_random]

In [ ]:
gnu = ld_prune(gn_fil, size=200, step=100, threshold=0.3, n_iter=5)

In [ ]:
rdm_idx = np.random.choice(gn_fil.shape[0], 500, replace=False)
plot_ld(gn_fil[rdm_idx], 'LD unpruned')

In [ ]:
rdm_idx = np.random.choice(gnu.shape[0], 500, replace=False)
plot_ld(gnu[rdm_idx], 'LD pruned')

## Plotting PCA

In [ ]:
# Run the PCA model
coords, model = allel.pca(gnu, n_components=10, scaler=None)

In [ ]:
# PLot PCA pairs

In [ ]:
fig, axes = plt.subplots(figsize=[9,8], ncols=2, nrows=2,
                         gridspec_kw={'top':0.97, 'right':0.84, 'bottom':0.08, 'left':0.09,
                                     'hspace':0.2, 'wspace':0.25})

for ax_idx, (pc1, pc2) in zip([(0,0), (0,1), (1,0), (1,1)], [(0,1), (2,3), (4,5), (6,7)]):

    ax = axes[ax_idx]

    x = coords[:, pc1]
    y = coords[:, pc2]
    
    dat = pd.concat([pd.Series(x), pd.Series(y), vcf_samples], axis=1)
    dat['Sample'] = dat[2].apply(lambda x: strain_alias[x])

    sns.scatterplot(x=0, y=1, hue='Sample', palette=Strain_color, hue_order=Strain_order, data=dat, ax=ax)
    for i in dat.index:
        ax.text(dat.loc[i, 0], dat.loc[i, 1], dat.loc[i, 'Sample'], size=5, color='k', alpha=0.5)

    ax.set_xlabel('PC%s (%.1f%%)' % (pc1+1, model.explained_variance_ratio_[pc1]*100))
    ax.set_ylabel('PC%s (%.1f%%)' % (pc2+1, model.explained_variance_ratio_[pc2]*100))

    if ax_idx == (1,1):
        ax.legend(loc=3, bbox_to_anchor=[1.1,0], frameon=False)
    else:
        ax.legend_.remove()

    sns.despine()

plt.savefig(f'{fig_path}PCA_quick.png', dpi=300)
#plt.show()
plt.close()

# Phylogenetic analysis

In [ ]:
with open(f'{base_dir}blastn/seq_id_list.txt') as handle:
    seq_id_list = handle.read().splitlines()

seq_id_keep = []

idx = 0
with progressbar.ProgressBar(max_value=len(seq_id_list)) as bar:
    for i in seq_id_list:
        
        blastn = pd.read_csv(f'{base_dir}blastn/blastn_tab/{i}.orf_coding.blastn.tab', sep='\t', header=None)
        blastn = blastn.set_index(1)
        
        if i in blastn.index:
            # define the bit-score threshold for self-matching blast hits 
            t_S = blastn.loc[i, 11] * 0.03
            if type(t_S) == pd.Series:
                t_S = t_S.min()
            # drop self-hits
            blastn = blastn.drop(i)
            
            if blastn.loc[blastn[11]> t_S].shape[0] <= 1:
                seq_id_keep.append(i)
            
        idx += 1
        bar.update(idx)

In [ ]:
np.random.seed(42)
seq_random_sel = np.random.choice(seq_id_keep, 100)
# Export in BED format
bed_random_sel = gff_genes.set_index('Name').loc[seq_random_sel]
bed_random_sel = pd.concat([bed_random_sel[0], bed_random_sel[3]-1, bed_random_sel[4]-1], axis=1)
bed_random_sel.to_csv(f'{base_dir}phylo/orf_coding.random100.bed', sep='\t', header=None, index=None)

# Export in GFF format
gff_random_sel = gff_genes.set_index('Name').loc[seq_random_sel, [0,1,2,3,4,5,6,7,'ID']]
gff_random_sel.to_csv(f'{base_dir}phylo/orf_coding.random100.gff', sep='\t', header=None, index=None)

## Import fasta for all strains and concat

In [ ]:
fasta_ids_order = gff_random_sel.sort_values(by=[0,3]).apply(lambda x: f'{x[0]}_{x[3]}-{x[4]}:{x[6]}', axis=1)

fasta = {s:Seq.Seq('') for s in strains_info.index}

for s in strains_info.index:
    
    with open(f'{base_dir}phylo/{s}.orf_coding.random100.fasta') as handle:
        fasta_dict = {seq.id:seq for seq in SeqIO.parse(handle, 'fasta')}
        
    for i in fasta_ids_order:
        if i in fasta_dict.keys():
            fasta[s] += fasta_dict[i].seq
        else:
            print(f'{i} {s} absent!!')
    
for s, seq in fasta.items():
    sr = SeqRecord.SeqRecord(seq=seq, id=s, description='')
    with open(f'{base_dir}phylo/{s}.orf_coding.random100.concat.fasta', 'w') as handle:
        SeqIO.write(sr, handle, 'fasta')

    seq_nonambig = seq
    for nt in ['R','W','Y','K','M','S']:
        seq_nonambig = seq_nonambig.replace(nt, 'N')
    sr = SeqRecord.SeqRecord(seq=seq_nonambig, id=s, description='')
    with open(f'{base_dir}phylo/{s}.orf_coding.random100.concat.nonambig.fasta', 'w') as handle:
        SeqIO.write(sr, handle, 'fasta')

In [ ]:
aln = AlignIO.read(f'{base_dir}phylo/orf_coding.random100.concat.nonambig.fasta', 'fasta')

f = []
for i in aln:
    f.append((np.array(i) == 'N').reshape(-1,1))

f = np.concatenate(f, axis=1).sum(axis=1)
f = ~(f > 1)

aln_fil = []
for sr in aln:
    sr2 = SeqRecord.SeqRecord(Seq.Seq(''.join(np.array(sr.seq)[f])), id=sr.id, description='')
    aln_fil.append(sr2)

with open(f'{base_dir}phylo/orf_coding.random100.concat.nonambig_trim.fasta', 'w') as handle:
        SeqIO.write(aln_fil, handle, 'fasta')

### Distance between Cg20 and Cg27

In [ ]:
aln = {seq.id:seq for seq in SeqIO.parse(f'{base_dir}phylo/orf_coding.random100.concat.fasta', 'fasta')}

In [ ]:
dist_phylo = pd.read_csv(f'{base_dir}phylo/orf_coding.random100.concat.dist.csv', index_col=0)
dist_phylo.loc['KSO_20', 'KSO_27']

In [ ]:
dist_dna = dist_phylo.copy()
dist_dna[:] = np.nan
for i,j in itertools.combinations(aln, 2):
    c = (np.array(aln[i].seq) != np.array(aln[j].seq)).sum()
    dist_dna.loc[i, j] = c
    dist_dna.loc[j, i] = c

In [ ]:
d1 = dist_phylo.values.flatten()
d2 = dist_dna.fillna(0).values.flatten()
lr = stats.linregress(d1, d2)
X = np.array([0, 6])*1e-3
plt.scatter(d1, d2)
plt.plot(X, X*lr.slope+lr.intercept)

# Diagnostic PCR

In [ ]:
restriction_candidates = {}
for s in ['KSO_138', 'KSO_4', 'KSO_20', 'KSO_27']:
    restriction_candidates[s] = {seq.id:seq for seq in SeqIO.parse(f'{base_dir}phylo/{s}.orf_coding.random100.fasta', 'fasta')}

In [ ]:
N_cuts = {}
for enzyme in Restriction.AllEnzymes:
    if enzyme.opt_temp == 37 and enzyme.is_defined():
        name = str(enzyme)
        N_cuts[name] = 0
        for s in restriction_candidates.keys():
            for i, seq in restriction_candidates[s].items():
                N_cuts[name] += len(enzyme.catalyse(seq.seq))-1

In [ ]:
N_cuts_AluI = []
for i, seq in restriction_candidates['KSO_138'].items():
    
    len_seq = len(seq.seq)
    ar = np.arange(0, len(seq), 200)
    ranges = [(ar[k], ar[k+5]) for k in range(ar.shape[0]-5)]
    for (start, end) in ranges:
        region_uid = f'{i}_{start}_{end}'
        for s in restriction_candidates:
            seq = restriction_candidates[s][i].seq[start:end]
            results = Restriction.AluI.catalyze(seq)
            for k, f in enumerate(results):
                N_cuts_AluI.append([region_uid, s, len_seq, k, len(f), str(f)])

N_cuts_AluI = pd.DataFrame(N_cuts_AluI, columns=['region_uid', 'strain', 'len_seq', 'fragment_idx', 'fragment_len', 'fragment_seq'])

In [ ]:
N_cuts_HinfI = []
for i, seq in restriction_candidates['KSO_138'].items():
    
    len_seq = len(seq.seq)
    ar = np.arange(0, len(seq), 200)
    ranges = [(ar[k], ar[k+5]) for k in range(ar.shape[0]-5)]
    for (start, end) in ranges:
        region_uid = f'{i}_{start}_{end}'
        for s in restriction_candidates:
            seq = restriction_candidates[s][i].seq[start:end]
            results = Restriction.HinfI.catalyze(seq)
            for k, f in enumerate(results):
                N_cuts_HinfI.append([region_uid, s, len_seq, k, len(f), str(f)])

N_cuts_HinfI = pd.DataFrame(N_cuts_HinfI, columns=['region_uid', 'strain', 'len_seq', 'fragment_idx', 'fragment_len', 'fragment_seq'])

In [ ]:
to_plot = []
for r, df in N_cuts_HinfI.groupby('region_uid'):
    len_distro = df.groupby('strain')['fragment_len'].apply(lambda x: set(np.round(x/1)))
    test_diff_len = [len_distro.loc[s1] == len_distro.loc[s2] for (s1, s2) in itertools.combinations(len_distro.index, 2)]
    if sum(test_diff_len) <= 1:
        to_plot.append(r)

In [ ]:
diagnostic_pcr = []
pcr_regions = {'PCR1':('ChrM_C_glabrata_CBS138_498770-501247:-', 1000-112-1, 2000+37),
               'PCR2':('ChrM_C_glabrata_CBS138_498770-501247:-', 1000-58-1, 2000+40),
               'PCR3':('ChrM_C_glabrata_CBS138_50036-56044:-', 2200-30-1, 3200+140),
               'PCR4':('ChrM_C_glabrata_CBS138_50036-56044:-', 2200-185-1, 3200+133)}
for pcr, (r, start, end) in pcr_regions.items():
    for s in restriction_candidates:
        seq = restriction_candidates[s][r].seq[start:end]
        diagnostic_pcr.append(SeqRecord.SeqRecord(seq, id=f'{pcr}_{s}', description=''))
SeqIO.write(diagnostic_pcr, f'{base_dir}sequences/diagnostic_pcr.fasta', 'fasta')

In [ ]:
fig, ax = plt.subplots(figsize=[6,6])
x = 0
xtick_labels = []
for seq in diagnostic_pcr:
    fragment_lengths = np.array([len(i) for i in Restriction.HinfI.catalyze(seq.seq)])
    for fl in fragment_lengths:
        ax.plot(np.array([-0.3, 0.3])+x, np.repeat(fl, 2), lw=2, c='w')
    s = strain_alias[seq.id[5:]]
    if type(s) == int:
        s = f'Cg{s}'
    pcr = seq.id[:4]
    xtick_labels.append(f'{s} {pcr}')
    x += 1
ax.set_facecolor('k')
ax.set_xticks(range(16))
ax.set_xticklabels(xtick_labels, rotation=90)

ax.set_ylabel('HinfI fragment size (bp)')

plt.tight_layout()
plt.savefig(f'{fig_path}diagnostic_pcr.png', dpi=300)
plt.show()
plt.close()

In [ ]:
for r in to_plot:
    
    df = N_cuts_HinfI.loc[N_cuts_HinfI['region_uid']==r]
    fig, ax = plt.subplots()
    sns.scatterplot(x='strain', y='fragment_len', marker='_', s=500, hue='fragment_idx', palette='viridis', data=df, ax=ax)
    ax.set_yscale('log')
    ax.legend(loc=3, bbox_to_anchor=[1,0])
    ax.set_title(r)
    plt.show()
    plt.close()
    

# Gene contents analysis

In [ ]:
# Text search on CGD for the term "adhesin": 70 hits in the "Description" field 
adhesins_cgd = pd.read_csv(f'{base_dir}script/CGD_adhesin.csv', skiprows=1, header=None)
adhesins_cgd_unique = []
for i in adhesins_cgd[1].values:
    adhesins_cgd_unique.extend(i.split(', '))
adhesins_cgd_unique = set(adhesins_cgd_unique)

# Genes list provided by Ost lab
genes_list_KSO = pd.read_excel(f'{base_dir}script/C.glabrata_Target_Gene_List.xlsx', skiprows=2)

# Gene list from the latest genome paper
gpi_adhesins_xu2020 = pd.read_csv(f'{base_dir}script/GPI_adhesins_xu2020.csv')

In [ ]:
adhesin_genes = set()
adhesin_genes_source = {}

print('### CGD ###')

for g in adhesins_cgd_unique:
    
    if g[:5] == 'CAGL0':
        df = gff_genes.loc[gff_genes['ID'] == g]
        if df.shape[0] == 0:
            df = gff_genes.loc[gff_genes['Alias'].apply(lambda x: g in x)]
        
    else:
        df = gff_genes.loc[gff_genes['Gene'] == g]
        if df.shape[0] == 0:
            df = gff_genes.loc[gff_genes['Alias'].apply(lambda x: g in x)]

    if df.shape[0] > 0:
        for i in df['ID'].values:
            adhesin_genes.add(i)
        
        if i not in adhesin_genes_source:
            adhesin_genes_source[i] = []
        adhesin_genes_source[i].append('cgd')

    else:
        print(f'** {g} NOT found **')

print(len(adhesin_genes))

print('### KSO lab ###')

for i in genes_list_KSO.index:
    g = genes_list_KSO.loc[i, 'Gene_ID']
    df = gff_genes.loc[gff_genes['ID'] == g]
    
    if df.shape[0] > 0:
        for i in df['ID'].values:
            adhesin_genes.add(i)
        
            if i not in adhesin_genes_source:
                adhesin_genes_source[i] = []
            adhesin_genes_source[i].append('kso')

    else:
        print(f'** {g} NOT found **')

print(len(adhesin_genes))

print('### Xu et al 2020 ###')

for i in gpi_adhesins_xu2020.index:
    g = gpi_adhesins_xu2020.loc[i, 'gene_id']
    df = gff_genes.loc[gff_genes['ID'] == g]
    
    if df.shape[0] > 0:
        for i in df['ID'].values:
            adhesin_genes.add(i)

            if i not in adhesin_genes_source:
                adhesin_genes_source[i] = []
            adhesin_genes_source[i].append('xu')

    else:
        print(f'** {g} NOT found **')

print(len(adhesin_genes))

In [ ]:
# export non-redundant final list to tsv for annotation
#gff_genes.set_index('ID').loc[sorted(adhesin_genes)].apply(lambda x: f'{x[0]}:{x[3]}-{x[4]}', axis=1)\
#.rename('region').to_csv(f'{base_dir}script/adhesin_genes.tsv', sep='\t')

In [ ]:
# Import the CGD query for adhesin gene IDs
adhesin_genes_annot = pd.read_csv(f'{base_dir}script/adhesin_genes_annot.tsv', sep='\t', header=None)
adhesin_genes_annot.index = adhesin_genes_annot[0].values

In [ ]:
gene_annot_order = ['adhesin',
                    'putative adhesin',
                    'aspartic protease',
                    'putative aspartic protease',
                    'hexose transporter',
                    'putative hexose transporter',
                    'dna binding',
                    'kinase']
gene_annot_color = ['mediumblue', 'dodgerblue', 'firebrick', 'lightcoral', 'forestgreen', 'lime', 'fuchsia', 'gold']
gene_annot_order_idx = dict(zip(gene_annot_order, range(8)))
adhesin_genes_annot['annot_order'] = adhesin_genes_annot[10].apply(lambda x: gene_annot_order_idx[x])
def generate_gene_tag(x):
    if x[0] == 'CAGL0J12067g':
        return(f"AWP11 | {x[0]}")
    elif pd.isna(x[1]):
        return(x[0])
    else:
        return(f"{x[1]} | {x[0]}")
adhesin_genes_annot['tag'] = adhesin_genes_annot.apply(lambda x: generate_gene_tag(x), axis=1)

adhesin_genes_annot = adhesin_genes_annot.sort_values(by=['annot_order', 0])
adhesin_genes_order = list(adhesin_genes_annot[0].values)

In [ ]:
adhesin_dp = []
strain_med_dp = {}

idx = 0

with progressbar.ProgressBar(max_value=len(adhesin_genes)*strains_info.shape[0]) as bar:

    for s in strains_info.index:
    
        path = f'../data/depth/{s}.depth.tab.gz'
        depth = pd.read_csv(path, sep='\t', header=None)
        strain_med_dp[s] = depth[2].median()
    
        for g in adhesin_genes:
            chrom, start, end = gff_genes.loc[g, [0, 3, 4]]
            g_med_dp = depth.loc[(depth[0]==chrom) & (depth[1]>=start) & (depth[1]<=end), 2].median()

            adhesin_dp.append([s, g, g_med_dp])
            
            idx += 1
            bar.update(idx)

In [ ]:
ADH_DP = pd.DataFrame(adhesin_dp)
ADH_DP.columns = ['strain', 'ID', 'med_dp']
ADH_DP['strain_dp'] = ADH_DP['strain'].apply(lambda x: strain_med_dp[x])
ADH_DP['dp'] = ADH_DP['med_dp']/ADH_DP['strain_dp']
ADH_DP['log_dp'] = np.log2(ADH_DP['dp'])
ADH_DP['log_dp'] = np.where(ADH_DP['log_dp']<-2, -2, ADH_DP['log_dp'])

In [ ]:
# Export ADH_DP
#ADH_DP.to_csv(f'{base_dir}tables/ADH_genes_depth.csv', index=False)
ADH_DP = pd.read_csv(f'{base_dir}tables/ADH_genes_depth.csv')

In [ ]:
fig = plt.figure(figsize=[12,7])
gs = plt.GridSpec(ncols=1, nrows=2, height_ratios=[1,19], wspace=0.15, hspace=0.05,
                 left=0.06, right=0.9, bottom=0.18, top=0.92)

dat = ADH_DP.pivot_table(index='strain', columns='ID', values='log_dp')

ax = fig.add_subplot(gs[1])
cbar_ax = fig.add_axes([0.92, 0.4, 0.015, 0.2])

S_phylo = [26,19,6,24,1,21,8,138,18,29,28,32,17,4,27,20,14,11,13,31,10,9,2,34,16,3,15,25,22,23,33,12,30]
S_phylo = [f'KSO_{s}' for s in S_phylo]

dat = dat.loc[S_phylo, adhesin_genes_order]

sns.heatmap(dat, ax=ax, cmap='bwr', center=0, cbar_ax=cbar_ax)

ax.set_ylabel('')
ax.set_yticklabels([strain_alias[s] for s in S_phylo])
ax.set_xlabel('')
ax.set_xticks(np.arange(len(adhesin_genes_order))+0.5)
ax.set_xticklabels(adhesin_genes_annot['tag'], rotation=90, size=6, ha='center')

# Highlight AWP11
for xt in ax.get_xticklabels():
    if xt.get_text() == 'AWP11 | CAGL0J12067g':
        xt.set_fontweight('bold')
        xt.set_color('limegreen')

r = Rectangle((69, 0), 1, len(S_phylo), fc=(1,1,1,0), ec='limegreen', lw=1.5, clip_on=False)
ax.add_patch(r)

cbar_ax.set_yticks(np.arange(-2,3))
cbar_ax.set_ylabel('relative copy number ($log_2$)')

# gene annot category

ax = fig.add_subplot(gs[0])

cmap = colors.ListedColormap(gene_annot_color)
bounds = np.arange(len(gene_annot_color)+1)
norm = colors.BoundaryNorm(bounds, cmap.N)

dat = adhesin_genes_annot['annot_order'].values.reshape(1, -1)
ax.imshow(dat, cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')
ax.axis('off')

legend_elms = [Line2D([0], [0], color='w', marker='s', mfc=c, ms=12, label=l) for (l,c) in zip(gene_annot_order, gene_annot_color)]
ax.legend(handles=legend_elms, ncol=4, loc=3, bbox_to_anchor=[0,1], frameon=False)


for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}hm_adhesin_cnv.{ext}', dpi=300)
#plt.show()
plt.close()

## Depth of the N-terminal domain of AWP11

In [ ]:
DP_AWP11_NTERM = []
for s in strains_info.index:

    path = f'../data/depth/{s}.depth.tab.gz'
    depth = pd.read_csv(path, sep='\t', header=None)
    dp_awp11_nterm = depth.set_index([0,1]).loc['ChrJ_C_glabrata_CBS138'].loc[1236526:1236526+1791, 2].median()
    DP_AWP11_NTERM.append([s, dp_awp11_nterm, strain_med_dp[s]])
    print(s)

DP_AWP11_NTERM = pd.DataFrame(DP_AWP11_NTERM, columns=['strain', 'med_dp', 'strain_dp'])

In [ ]:
DP_AWP11_NTERM['dp'] = DP_AWP11_NTERM['med_dp']/DP_AWP11_NTERM['strain_dp']
DP_AWP11_NTERM['log_dp'] = np.log2(DP_AWP11_NTERM['dp'])
DP_AWP11_NTERM['log_dp'] = np.where(DP_AWP11_NTERM['log_dp']<-2, -2, DP_AWP11_NTERM['log_dp'])
DP_AWP11_NTERM['fl_dp'] = ADH_DP.set_index(['ID', 'strain']).loc['CAGL0J12067g'].loc[DP_AWP11_NTERM['strain'].values, 'dp'].values

In [ ]:
sns.scatterplot(x='dp', y='fl_dp', hue='strain', data=DP_AWP11_NTERM)

In [ ]:
S = [f'@{i}' for i in (26,19,6,24,1,21,8)] + ['$138'] +\
[f'@{i}' for i in (18,29,28,32,17,4,27,20,14,11,13,31,10,9,2,34,16,3,15,25,22,23,33,12,30)]
S_Cg = [s.replace('@', 'Cg').replace('$', 'CBS') for s in S]
S_KSO = [s.replace('@', 'KSO_').replace('$', 'KSO_') for s in S]

fig, ax = plt.subplots(figsize=[3,7])
ax.barh(range(33), DP_AWP11_NTERM.set_index('strain').loc[S_KSO, 'dp'], color='k')
ax.set_yticks(range(33))
ax.set_yticklabels(S_Cg)
ax.invert_yaxis()
ax.set_xlabel('$AWP11$ N-terminal copy number')

sns.despine()
plt.tight_layout()
plt.savefig(f'{fig_path}AWP11_Nterminal_CN.svg')
#plt.show()
plt.close()

# Long-read assemblies

In [ ]:
S_lr = ['CBS138', 'Cg1', 'Cg14', 'Cg27']

In [ ]:
# Compute N50 for Canu assemblies
canu_assemblies = {}
for s in S_lr:
    canu_assemblies[s] = {seq.id:seq for seq in SeqIO.parse(f'{base_dir}canu/{s}_defaults.bk/{s}.contigs.fasta', 'fasta')}

In [ ]:
canu_n50 = {}
for s in S_lr:
    cl = np.sort(np.array([len(seq.seq) for seq in canu_assemblies[s].values()]))[::-1]
    n50 = cl[np.cumsum(cl) > 0.5*cl.sum()][0]
    canu_n50[s] = n50

In [ ]:
LR_len = {}
LR_N50 = {}
for s in S_lr:
    rl_path = f'{base_dir}data/fastq_nanopore/{s}.readlengths.txt'
    rl = pd.read_csv(rl_path, header=None)[0].sort_values(ascending=False)
    LR_N50[s] = rl.loc[np.cumsum(rl)>0.5*rl.sum()].iloc[0]
    LR_len[s] = rl

for s in pd.read_csv('/home/mathieu/postdoc_heasley/long_read_project/script/strains_info.csv').iloc[-9:]['strain'].values:
    rl_path = f'/home/mathieu/data/minknow/trimmed/{s}_readlengths.txt'
    rl = pd.read_csv(rl_path, header=None)[0].sort_values(ascending=False)
    LR_N50[s] = rl.loc[np.cumsum(rl)>0.5*rl.sum()].iloc[0]
    LR_len[s] = rl

In [ ]:
fig, ax = plt.subplots(figsize=[6,6])
for s, rl in LR_len.items():
    if s in S_lr:
        c = 'red'
        lw = 2
        l = s
    else:
        c = 'k'
        lw = 0.5
        l = ''

    #ax.plot(np.log10(rl), np.linspace(100, 0, rl.shape[0]), c=c, lw=lw)
    rl = rl.sort_values(ascending=True)
    ax.plot(np.log10(rl), np.cumsum(rl), c=c, lw=lw)
    n50 = np.log10(LR_N50[s])
    ax.axvline(n50, c=c, lw=lw, ls='--')
    ax.text(n50, 107e6, l, rotation=90, c='red', ha='center', va='bottom')

ax.set_xlim(1.5, 5.5)
ax.set_xlabel('log10 read size')
ax.set_ylabel('cumul %')

plt.tight_layout()
#plt.savefig(f'{fig_path}read_size_distro.png', dpi=300)
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=[10,10])

for i, (s,d) in enumerate(LR_len.items()):
    d = np.sort(np.array(d))
    ax.plot(d, np.arange(0, d.shape[0]), label=s)
    m = d[d<np.median(d)][-1]
    ax.axvline(m, ls=':', c='k', lw=0.5)
    ax.text(m, d.shape[0]*0.5, f'{s} = {m}', ha='left', va='center')

ax.set_xscale('log')

plt.legend()
plt.show()
plt.close()

In [ ]:
Assembly = {}
Tig_Off_Assembly = {}
Coords = {}
Ctg_Assign = {}

for s in S_lr:
    for a in ['canu', 'shasta', 'wtdbg2']:

        k = f'{s}_{a}'
        
        assembly = {seq.id:seq for seq in SeqIO.parse(f'{base_dir}mummer/{k}_raw.fasta', 'fasta')}
        Assembly[k] = assembly
        
        coords = pd.read_csv(f'{base_dir}mummer/{k}_raw.ref.coords', sep='\t', skiprows=4, header=None)
        coords = coords.loc[(coords[4]>5000) & (coords[5]>5000)]
        coords[10] = coords[10].astype(str)
        
        ctg_assign = coords.sort_values(by=5, ascending=False).groupby(10).apply(lambda x: x.iloc[0], include_groups=False)
        ctg_assign['midpoint'] = ctg_assign[[0,1]].mean(axis=1)
        ctg_assign = ctg_assign.sort_values(by=[9, 'midpoint'])

        Ctg_Assign[k] = ctg_assign
        
        tig_off_assembly = pd.DataFrame({i:len(seq.seq) for i,seq in assembly.items()}, index=['len']).T.loc[ctg_assign.index]
        
        tig_off_assembly['cumul_end'] = np.cumsum(tig_off_assembly['len'])
        tig_off_assembly['cumul_start'] = np.concatenate([np.array([0]), tig_off_assembly['cumul_end'].values[:-1]])
        Tig_Off_Assembly[k] = tig_off_assembly
        
        coords['sstart'] = coords[0]
        coords['send'] = coords[1]
        
        for ctg, df in coords.groupby(10):
            strand = ctg_assign.loc[ctg, 8]
            if strand == -1:
                coords.loc[df.index, 'qstart'] = np.abs(coords[2] - tig_off_assembly.loc[ctg, 'len'])
                coords.loc[df.index, 'qend'] = np.abs(coords[3] - tig_off_assembly.loc[ctg, 'len'])
        
            else:
                coords.loc[df.index, 'qstart'] = coords[2]
                coords.loc[df.index, 'qend'] = coords[3]
        coords['strand'] = np.where(coords['qstart'] < coords['qend'], '+', '-')
        
        coords['Qstart'] = coords['qstart'] + tig_off_assembly.loc[coords[10], 'cumul_start'].values
        coords['Qend'] = coords['qend'] + tig_off_assembly.loc[coords[10], 'cumul_start'].values
        
        coords['Sstart'] = coords['sstart'] + tig_off.loc[coords[9], 'cumul_start'].values
        coords['Send'] = coords['send'] + tig_off.loc[coords[9], 'cumul_start'].values
        
        Coords[k] = coords

In [ ]:
# export reordered genomes
for k, ctg_assign in Ctg_Assign.items():
    new_genome = []
    for ctg in ctg_assign.index:
        
        seq = Assembly[k][ctg].seq
        orient = 'plus'
        if ctg_assign.loc[ctg, 8] == -1:
            seq = seq.reverse_complement()
            orient = 'minus'

        new_sr = SeqRecord.SeqRecord(seq=seq, id=f'{ctg}_{orient}', description='')
        new_genome.append(new_sr)

        with open(f'{base_dir}reordered_assemblies/{k}_reordered.fasta', 'w') as handle:
            #SeqIO.write(new_genome, handle, 'fasta')
            ...

In [ ]:
Assembly_Reordered = {}

for s in S_lr:
    for a in ['canu', 'shasta', 'wtdbg2']:

        k = f'{s}_{a}'
        
        assembly = {seq.id:seq for seq in SeqIO.parse(f'{base_dir}reordered_assemblies/{k}_reordered.fasta', 'fasta')}
        Assembly_Reordered[k] = assembly

## Export coords in bed format for genome browsing

In [ ]:
Coords_reordered = {}

for s in S_lr:
    for a in ['canu', 'shasta', 'wtdbg2']:
        k = f'{s}_{a}'
        coords = pd.read_csv(f'{base_dir}mummer/{k}_reordered.ref.coords', sep='\t', skiprows=4, header=None)
        coords = coords.loc[(coords[4]>5000) & (coords[5]>5000)]
        coords[10] = coords[10].astype(str)
        
        coords['chrom'] = coords[9]
        coords['start'] = coords[0]-1
        coords['end'] = coords[1]
        coords['name'] = coords.apply(lambda x: f'{x[10].split("_")[0]}_{x[2]}_{x[3]}', axis=1)
        coords['score'] = '.'
        coords['strand'] = coords[8].replace({1:'+', -1:'-'})
        
        coords[['chrom', 'start', 'end', 'name', 'score', 'strand']].to_csv(f'{base_dir}reordered_assemblies/{k}_reordered.ref.bed', 
                                                                           sep='\t', header=False, index=False)

In [ ]:
for s in S_lr:

    #for a in ['canu', 'shasta', 'wtdbg2']:
    for a in ['canu']:
    #a = 'canu'
        k = f'{s}_{a}'
        
        coords = Coords[k]
        tig_off_assembly = Tig_Off_Assembly[k]
        ctg_assign = Ctg_Assign[k]
        
        fig, ax = plt.subplots(figsize=[10, 10])
        
        for i in coords.index:
            ctg = coords.loc[i, 10]

            if ctg_assign.loc[ctg, 8] == -1:
                c = 'aqua'
            else:
                c = 'black'
            X = coords.loc[i, ['Sstart', 'Send']].values
            Y = coords.loc[i, ['Qstart', 'Qend']].values
            ax.plot(X, Y, c=c, lw=2, zorder=1)
        
        max_x = tig_off['cumul_end'].iloc[-1]
        max_y = tig_off_assembly['cumul_end'].iloc[-1]
        ax.set_xlim(0, max_x)
        ax.set_ylim(0, max_y)
        
        for ctg in tig_off_assembly.iloc[:-1].index:
            ax.axhline(tig_off_assembly.loc[ctg, 'cumul_end'], lw=0.5, zorder=0, c='k', ls=':')
            if tig_off_assembly.loc[ctg, 'len'] > 25e4:
                y1, y2 = tig_off_assembly.loc[ctg, ['cumul_start', 'cumul_end']]
                ax.text(max_x*1.02, np.mean([y1, y2]), ctg, size=8)
        
        for chrom in tig_off.iloc[:-1].index:
            x1, x2 = tig_off.loc[chrom, ['cumul_start', 'cumul_end']]
            ax.axvline(x1, lw=0.5, zorder=0, c='k', ls=':')
            ax.text(np.mean([x1, x2]), max_y*1.02, chrom.split('_')[0], size=8, rotation=90)
            if chrom in centromeres.index:
                ax.axvline(centromeres.loc[chrom, 'Loc'], lw=3, c='blue', alpha=0.2, zorder=0)
        
        xticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])
        xtick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])*1e-3
        ax.set_xticks(xticks)
        ax.set_xticklabels([f'{x:.0f}' if x>0 else '' for x in xtick_labels], size=8, rotation=90)
        ax.set_xlabel('CBS138 reference genome (kb)')
        
        yticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iloc[:-1].iterrows()])
        ytick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iloc[:-1].iterrows()])*1e-3
        ax.set_yticks(yticks)
        ax.set_yticklabels([f'{y:.0f}' if y>0 else '' for y in ytick_labels], size=8)
        ax.set_ylabel('Assembly (kb)')
        
        ax.set_title(f'{s} {a} assembly')
        
        plt.tight_layout()
        #plt.savefig(f'{fig_path}dotplot_{k}.png', dpi=300)
        plt.show()
        plt.close()

In [ ]:
#ctg_alt_color = dict(zip(range(10), colorcet.glasbey_light[:10]))
ctg_alt_color = {0:'0', 1:'0.7'}

s = 'Cg27'
a = 'canu'
k = f'{s}_{a}'

coords = Coords[k].copy()
ctg_assign = Ctg_Assign[k].copy()
ctg_assign = ctg_assign.loc[ctg_assign[9]!='mito_C_glabrata_CBS138']
coords = coords.loc[coords[10].isin(ctg_assign.index)]
tig_off_assembly = Tig_Off_Assembly[k].copy()
tig_off_assembly = tig_off_assembly.loc[ctg_assign.index]
n_ctg = tig_off_assembly.shape[0]
tig_off_assembly['alt'] = np.tile(range(2), n_ctg)[:n_ctg]

fig, ax = plt.subplots(figsize=[7,7])

for ctg in tig_off_assembly.index:
    sub_coords = coords.loc[coords[10]==ctg]
    c = ctg_alt_color[tig_off_assembly.loc[ctg, 'alt']]
    for i in sub_coords.index:
        X = sub_coords.loc[i, ['Sstart', 'Send']].values
        Y = sub_coords.loc[i, ['Qstart', 'Qend']].values
        ax.plot(X, Y, c=c, lw=4, zorder=1)

max_x = tig_off['cumul_end'].iloc[-1]
max_y = tig_off_assembly['cumul_end'].iloc[-1]
ax.set_xlim(0, max_x)
ax.set_ylim(0, max_y)

for ctg in tig_off_assembly.iloc[:-1].index:
    ax.axhline(tig_off_assembly.loc[ctg, 'cumul_end'], lw=0.5, zorder=0, c='k', ls=':')
    """if tig_off_assembly.loc[ctg, 'len'] > 25e4:
        y1, y2 = tig_off_assembly.loc[ctg, ['cumul_start', 'cumul_end']]
        ax.text(max_x*1.02, np.mean([y1, y2]), ctg, size=8)"""

y_cen_mark = tig_off_assembly.iloc[-1]['cumul_end']
for chrom in tig_off.iloc[:-1].index:
    x1, x2 = tig_off.loc[chrom, ['cumul_start', 'cumul_end']]
    ax.axvline(x1, lw=0.5, zorder=0, c='k', ls=':')
    if chrom in centromeres.index:
        ax.scatter(centromeres.loc[chrom, 'Loc'], y_cen_mark, marker='|', c='blue', clip_on=False, zorder=2)

xticks = np.concatenate([np.arange(cs, ce, 5e5) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])
xtick_labels = np.concatenate([np.arange(0, l, 5e5) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])*1e-6
ax.set_xticks(xticks)
ax.set_xticklabels([f'{x:.1f}' if x>0 else '' for x in xtick_labels], size=8, rotation=90)
ax.set_xlabel('CBS138 reference genome (Mb)', size=14)

yticks = np.concatenate([np.arange(cs, ce, 5e5) for chrom, (l, ce, cs, al) in tig_off_assembly.iterrows()])
ytick_labels = np.concatenate([np.arange(0, l, 5e5) for chrom, (l, ce, cs, al) in tig_off_assembly.iterrows()])*1e-6
ax.set_yticks(yticks)
ax.set_yticklabels([f'{y:.1f}' if y>0 else '' for y in ytick_labels], size=8)
ax.set_ylabel(f'{s} assembly (Mb)', size=14)

#plt.tight_layout()
#plt.savefig(f'{fig_path}dotplot_{k}_simplified.svg')
plt.show()
plt.close()

# BLAST of CAGL0J12067g

## Self-alignment

In [ ]:
blastn_self = pd.read_csv(f'{base_dir}sequences/CAGL0J12067g.self_dna.blastn.tab', sep='\t', header=None)
blastp_self = pd.read_csv(f'{base_dir}sequences/CAGL0J12067g.self_protein.blastp.tab', sep='\t', header=None)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=[16,8])

ax = axes[0]
for i in blastn_self.index:
    x1, x2, y1, y2, pid = blastn_self.loc[i, [6,7,8,9,2]]
    ax.plot([x1,x2], [y1,y2], c=f'{(100-pid)/100*2}')

ax = axes[1]
for i in blastp_self.index:
    x1, x2, y1, y2, pid = blastp_self.loc[i, [6,7,8,9,2]]
    ax.plot([x1,x2], [y1,y2], c=f'{(100-pid)/100*2}')


plt.show()
plt.close()

In [ ]:
blastn = pd.read_csv(f'{base_dir}sequences/CAGL0J12067g.Cg27_canu.blastn.tab', sep='\t', header=None)

In [ ]:
for tig, df in blastn.groupby(1):
    fig, ax = plt.subplots()
    for i in df.index:
        x1, x2, y1, y2, pid = df.loc[i, [6,7,8,9,2]]
        ax.plot([x1,x2], [y1,y2], c=f'{(100-pid)/100*2}')
    ax.set_title(tig)
    plt.show()
    plt.close()

In [ ]:
blastn_self = pd.read_csv(f'{base_dir}sequences/CAGL0J12067g.self.blastn.tab', sep='\t', header=None)

In [ ]:
for tig, df in blastn_self.groupby(1):
    fig, ax = plt.subplots()
    for i in df.index:
        x1, x2, y1, y2, pid = df.loc[i, [6,7,8,9,2]]
        ax.plot([x1,x2], [y1,y2], c=f'{(100-pid)/100}')
    ax.set_title(tig)
    ax.set_xticks(np.arange(0, 7000, 500))
    plt.show()
    plt.close()

## Blast to bed

In [ ]:
qpcr_primers = {seq.id:seq for seq in SeqIO.parse(f'{base_dir}sequences/021324_Cg_qPCR_primers.fasta', 'fasta')}

In [ ]:
for s in ['Cg1', 'Cg14', 'Cg27']:
    
    blastn_primers = pd.read_csv(f'{base_dir}sequences/qPCR_primers.{s}_canu.blastn.tab', sep='\t', header=None)
    blastn_primers['primer_len'] = blastn_primers[0].apply(lambda x: len(qpcr_primers[x].seq))
    blastn_primers = blastn_primers.loc[blastn_primers[7] == blastn_primers['primer_len']]
    blastn_primers['chr'] = blastn_primers[1]
    blastn_primers['start'] = blastn_primers[[8,9]].min(axis=1)-1
    blastn_primers['end'] = blastn_primers[[8,9]].max(axis=1)-1
    blastn_primers['name'] = blastn_primers[0]
    blastn_primers['score'] = blastn_primers[2]*10
    blastn_primers['strand'] = np.where(blastn_primers[8]<blastn_primers[9], '+', '-')
    blastn_primers[['chr','start','end','name','score','strand']].to_csv(f'{base_dir}sequences/qPCR_primers.{s}_canu.blastn.bed',
                                                                         sep='\t', index=False, header=False)

    blastn_5pd = pd.read_csv(f'{base_dir}sequences/adhesins_5prime_domains.{s}_canu.tblastn.tab', sep='\t', header=None)
    blastn_5pd = blastn_5pd.loc[blastn_5pd[3]>=150]
    blastn_5pd['chr'] = blastn_5pd[1]
    blastn_5pd['start'] = blastn_5pd[[8,9]].min(axis=1)-1
    blastn_5pd['end'] = blastn_5pd[[8,9]].max(axis=1)-1
    blastn_5pd['name'] = blastn_5pd[0]
    blastn_5pd['score'] = blastn_5pd[2]*10
    blastn_5pd['strand'] = np.where(blastn_5pd[8]<blastn_5pd[9], '+', '-')
    blastn_5pd[['chr','start','end','name','score','strand']].to_csv(f'{base_dir}sequences/adhesins_5prime_domains.{s}_canu.tblastn.bed',
                                                                         sep='\t', index=False, header=False)

## BLASTN of CAGL0J12067g against the new reordered assemblies

In [ ]:
blastn_CAGL0J12067g = []

for s in S_lr:
    for a in ['canu', 'shasta', 'wtdbg2']:
        blastn = pd.read_csv(f'{base_dir}blastn_CAGL0J12067g/{s}_{a}_CAGL0J12067g.blastn.tab', sep='\t', header=None)
        blastn['strain'] = s
        blastn['assembly'] = a

        blastn_CAGL0J12067g.append(blastn)

blastn_CAGL0J12067g = pd.concat(blastn_CAGL0J12067g).reset_index(drop=True)

In [ ]:
cluster_idx = 0
for (ctg, s, a), df in blastn_CAGL0J12067g.groupby([1, 'strain', 'assembly']):
    dat = np.sort(df[[8,9]].values, axis=1)
    coords = []
    for i in dat:
        coords.extend(np.arange(i[0], i[1], 250))
    coords = np.unique(coords)
    
    coords_cluster = [cluster_idx]
    for i in np.arange(coords.shape[0]-1):
        diff = coords[i+1]-coords[i]
        if diff > 500:
            cluster_idx += 1
        coords_cluster.append(cluster_idx)
    
    coords_cluster_dict = dict(zip(coords, coords_cluster))
    blastn_CAGL0J12067g.loc[df.index, 'cluster'] = np.sort(df[[8,9]].map(lambda x: coords_cluster_dict.get(x, np.nan)).values, axis=1)[:, 0]
    
    cluster_idx += 1

In [ ]:
# putative real hits: those that include the 5' portion of the gene

CAGL0J12067g_hits = []
CAGL0J12067g_hits_flanking = []

# Open reference sequences

with open(f'{base_dir}sequences/CAGL0J12067g.dna.fasta') as fi:
    sr = SeqIO.read(fi, 'fasta')
    CAGL0J12067g_hits.append(sr)

with open(f'{base_dir}sequences/CAGL0J12067g.dna_flanking.fasta') as fi:
    sr = SeqIO.read(fi, 'fasta')
    CAGL0J12067g_hits_flanking.append(sr)
    
for (cl, s, a, ctg), df in blastn_CAGL0J12067g.sort_values(by=[6,7]).groupby(['cluster', 'strain', 'assembly', 1]):

    if df[6].min() < 100:
        fig, ax = plt.subplots(figsize=[4,4])
        for i in df.index:
            ax.plot(df.loc[i, [6,7]], df.loc[i, [8,9]])

        ax.set_title(f'{cl} {s} {a} {ctg}')
        ax.set_xlim(0, 6336)
        plt.show()
        plt.close()

        reverse = (df[9]-df[8]).mean() < 0
        coords = np.sort(df[[8,9]].values.flatten())
        start, end = coords[0], coords[-1]

        # export sequences without flank
        name = f'{s}_{a}_{ctg}_{start}_{end}{"_rev" if reverse else ""}'
        seq = Assembly_Reordered[f'{s}_{a}'][ctg].seq[start-1:end]
        if reverse:
            seq = seq.reverse_complement()
        sr = SeqRecord.SeqRecord(seq=seq, id=name, description='')
        CAGL0J12067g_hits.append(sr)
        

        # export sequences without flank
        start -= 1000
        end += 1000
        name = f'{s}_{a}_{ctg}_{start}_{end}{"_rev" if reverse else ""}'
        seq = Assembly_Reordered[f'{s}_{a}'][ctg].seq[start-1:end]
        if reverse:
            seq = seq.reverse_complement()
        sr = SeqRecord.SeqRecord(seq=seq, id=name, description='')
        CAGL0J12067g_hits_flanking.append(sr)

with open(f'{base_dir}blastn_CAGL0J12067g/CAGL0J12067g_hits.fasta', 'w') as handle:
    #SeqIO.write(CAGL0J12067g_hits, handle, 'fasta')
    ...

with open(f'{base_dir}blastn_CAGL0J12067g/CAGL0J12067g_hits_flanking.fasta', 'w') as handle:
    #SeqIO.write(CAGL0J12067g_hits_flanking, handle, 'fasta')
    ...

# Dotplot figures

In [ ]:
for s in S_lr:

    #for a in ['canu', 'shasta', 'wtdbg2']:
    for a in ['canu']:
    #a = 'canu'
        k = f'{s}_{a}'
        
        coords = Coords[k]
        tig_off_assembly = Tig_Off_Assembly[k]
        ctg_assign = Ctg_Assign[k]
        blastn_sub = blastn_CAGL0J12067g.set_index(['strain', 'assembly']).loc[(s, a)].reset_index()
        
        fig, ax = plt.subplots(figsize=[10, 10])
        
        for i in coords.index:
            ctg = coords.loc[i, 10]

            #if ctg_assign.loc[ctg, 8] == -1:
            #    c = 'aqua'
            #else:
            #    c = 'black'
            
            X = coords.loc[i, ['Sstart', 'Send']].values
            Y = coords.loc[i, ['Qstart', 'Qend']].values

            if Y[0] > Y[1]:
                c = 'red'
            else:
                c = 'k'
            
            ax.plot(X, Y, c=c, lw=2, zorder=1)
        
        max_x = tig_off['cumul_end'].iloc[-1]
        max_y = tig_off_assembly['cumul_end'].iloc[-1]
        ax.set_xlim(0, max_x)
        ax.set_ylim(0, max_y)
        
        for ctg in tig_off_assembly.index:
            ax.axhline(tig_off_assembly.loc[ctg, 'cumul_end'], lw=0.5, zorder=0, c='k', ls=':')
            if tig_off_assembly.loc[ctg, 'len'] > 25e4:
                y1, y2 = tig_off_assembly.loc[ctg, ['cumul_start', 'cumul_end']]
                ax.text(max_x*1.02, np.mean([y1, y2]), ctg, size=8)
        
        yticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iterrows()])
        ytick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in tig_off_assembly.iterrows()])*1e-3
        ax.set_yticks(yticks)
        ax.set_yticklabels([f'{y:.0f}' if y>0 else '' for y in ytick_labels], size=8)
        ax.set_xlabel('Assembly (kb)')
        
        for chrom in tig_off.iloc[:-1].index:
            x1, x2 = tig_off.loc[chrom, ['cumul_start', 'cumul_end']]
            ax.axvline(x1, lw=0.5, zorder=0, c='k', ls=':')
            ax.text(np.mean([x1, x2]), max_y*1.02, chrom.split('_')[0], size=8, rotation=90)
            if chrom in centromeres.index:
                ax.axvline(centromeres.loc[chrom, 'Loc'], lw=3, c='blue', alpha=0.2, zorder=0)
        
        xticks = np.concatenate([np.arange(cs, ce, 25e4) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])
        xtick_labels = np.concatenate([np.arange(0, l, 25e4) for chrom, (l, ce, cs) in tig_off.iloc[:-1].iterrows()])*1e-3
        ax.set_xticks(xticks)
        ax.set_xticklabels([f'{x:.0f}' if x>0 else '' for x in xtick_labels], size=8, rotation=90)
        ax.set_xlabel('CBS138 reference genome (kb)')

        # Add CAGL0J12067g annotations in the reference
        x = tig_off.loc['ChrJ_C_glabrata_CBS138', 'cumul_start'] + 1239694.5
        ax.axvline(x, c='limegreen', lw=3, alpha=0.5)

        # Add CAGL0J12067g annotations in the assembly
        for (cl, ctg), df in blastn_sub.groupby(['cluster', 1]):
            ctg = re.match('(.+)_(plus|minus)', ctg).group(1)
            if df[6].min() < 100:
                
                hit = np.sort(df[[8,9]].values.flatten())
                y = tig_off_assembly.loc[ctg, 'cumul_start'] + hit[[0, -1]].mean()
                ax.axhline(y, c='limegreen', lw=3, alpha=0.5)
        
        ax.set_title(f'{s} {a} assembly')
        
        plt.tight_layout()
        #plt.savefig(f'{fig_path}dotplot_{k}.png', dpi=300)
        plt.show()
        plt.close()

# AWP11 5' paralogs

In [ ]:

awp11_aa = []

for s in strains_info.index:
    haplotypes_dna = []
    for h in (1,2):
        sr = SeqIO.read(f'{base_dir}whatshap/{s}_CAGL0J12067g.H{h}.fasta', 'fasta')
        sr.id = f'{s}_CAGL0J12067g.H{h}'
        haplotypes_dna.append(sr)
    haplotypes_aa = []
    for sr in haplotypes_dna:
        srt = sr.translate()
        srt.id = sr.id
        srt.description=''
        haplotypes_aa.append(srt)
    
    if haplotypes_aa[0].seq == haplotypes_aa[1].seq:
        del haplotypes_aa[1]
    awp11_aa.extend(haplotypes_aa)

SeqIO.write(awp11_aa, f'{base_dir}sequences/AWP11_aa.fasta', 'fasta')